# 06 · `dim_campana` y `fact_campanas` — contactos y atribución de ventas
**IDY1102 · Experiencia 3 · DataRetail 360**

| Campo | Detalle |
|---|---|
| Responsable | [Victor Navarrete] |
| Tablas destino | `dim_campana` (SCD1) y `fact_campanas` |
| Grano | una fila por **cliente contactado × campaña × canal × fecha de envío × producto promocionado** |
| Medidas | `contactado`, `respondio`, `compro`, `monto_venta_atribuida`, `costo_asignado` |
| Depende de | `fact_ventas` y las dimensiones (notebooks 01 a 05) |
| Fuentes | `campanas.csv` y `campanas_contactos.csv` (Marketing) |
| Alimenta a | K5 (efectividad de campaña, ROAS) de la ficha de KPI |

> **Marketing no informa ventas.** Lo que compró cada cliente se calcula aquí con `fact_ventas` y una **regla de atribución**
> explícita. Esa regla es una decisión de negocio y está documentada en la sección 7.

## 0. Setup

In [ ]:
import os, sys
from datetime import date
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src import db, extract, medallion, quality as q, schema
from src.config import get_settings
from src.logger import get_logger

PROCESO = "fact_campanas"
log = get_logger(PROCESO)
cfg = get_settings()
engine = db.get_engine(cfg)
schema.crear_esquema(engine)                  # idempotente: crea solo lo que falta
schema.sembrar_miembros_desconocidos(engine)  # miembro -1 "No informado" en cada dimensión
pd.set_option("display.width", 160)

## 1. Parámetros de negocio

In [ ]:
# Regla de atribución: una venta se atribuye al contacto cuando el MISMO cliente compra el MISMO producto
# promocionado dentro de N días desde el envío. Si cae en la ventana de varias campañas, gana la más reciente.
VENTANA_ATRIBUCION_DIAS = 14

## A. `dim_campana` (SCD Tipo 1)

### A.1 Extract

In [ ]:
def extraer_campanas() -> pd.DataFrame:
    df = extract.leer_csv(cfg.data_dir / "campanas.csv")
    medallion.guardar_capa(df, "bronze", "campanas")
    return df

### A.2 Transform
Reglas: código obligatorio y único (Unicidad), presupuesto positivo (Validez), fechas válidas y coherentes (Validez).

In [ ]:
COLUMNAS_CAMPANA = ["codigo_campana", "nombre_campana", "tipo_campana", "objetivo", "fecha_inicio", "fecha_fin", "presupuesto"]

def transformar_campanas(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    df = df.copy()
    rechazos = []
    df["codigo_campana"] = q.normalizar_texto(df["codigo_campana"], mayusculas=True)
    df, r = q.separar_rechazos(df, df["codigo_campana"].isna() | (df["codigo_campana"] == ""), "codigo_vacio"); rechazos.append(r)
    df["presupuesto"] = pd.to_numeric(df["presupuesto"], errors="coerce").astype("float64")
    df, r = q.separar_rechazos(df, ~(df["presupuesto"] > 0), "presupuesto_invalido"); rechazos.append(r)
    for col in ["fecha_inicio", "fecha_fin"]:
        df[col] = q.parsear_fechas(df[col])
    df, r = q.separar_rechazos(df, df["fecha_inicio"].isna() | df["fecha_fin"].isna() | (df["fecha_fin"] < df["fecha_inicio"]),
                               "fechas_invalidas"); rechazos.append(r)
    df, r = q.separar_rechazos(df, df["codigo_campana"].duplicated(keep="first"), "campana_duplicada"); rechazos.append(r)
    for col in ["nombre_campana", "tipo_campana", "objetivo"]:
        df[col] = q.normalizar_texto(df[col])
    return df[COLUMNAS_CAMPANA + ["origen_archivo", "fila_origen"]], q.consolidar_rechazos(*rechazos)

### A.3 Load

In [ ]:
def cargar_dim_campana(df_limpio: pd.DataFrame, df_rechazos: pd.DataFrame) -> dict:
    db.registrar_rechazos(engine, "dim_campana", df_rechazos)
    return db.aplicar_scd1(engine, "dim_campana", df_limpio[COLUMNAS_CAMPANA], "codigo_campana",
                           ["nombre_campana", "tipo_campana", "objetivo", "fecha_inicio", "fecha_fin", "presupuesto"],
                           "campana_key")

with db.Ejecucion(engine, "dim_campana") as run:
    campanas_crudo = extraer_campanas()
    run.leidas = len(campanas_crudo)
    campanas_limpias, campanas_rechazos = transformar_campanas(campanas_crudo)
    resumen_campanas = cargar_dim_campana(campanas_limpias, campanas_rechazos)
    run.cargadas = resumen_campanas["nuevos"]
    run.rechazadas = len(campanas_rechazos)
resumen_campanas

## B. `fact_campanas`

### B.1 Extract

In [ ]:
def extraer_contactos() -> pd.DataFrame:
    df = extract.leer_csv(cfg.data_dir / "campanas_contactos.csv")
    medallion.guardar_capa(df, "bronze", "campanas_contactos")
    return df

### B.2 Transform
| Problema | Dimensión de calidad | Regla |
|---|---|---|
| RUT con dígito verificador incorrecto | Validez | `q.validar_ruts` |
| El mismo cliente contactado dos veces en una campaña | Unicidad | un contacto por `(campaña, cliente)` |
| Campaña o SKU que no existen | Integridad referencial | se rechazan |
| Cliente sin ficha en `dim_cliente` | Integridad referencial | se rechaza (`cliente_no_registrado`) |
| Envío fuera de las fechas de la campaña | Consistencia | se rechaza (`fuera_de_campana`) |
| Canal escrito de varias formas | Consistencia | `q.canal_canonico` |

In [ ]:
def transformar_contactos(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    from src.config import exigir_rut_salt
    df = df.copy()
    rechazos = []

    df["fecha_envio"] = q.parsear_fechas(df["fecha_envio"])
    df, r = q.separar_rechazos(df, df["fecha_envio"].isna(), "fecha_invalida"); rechazos.append(r)
    df, r = q.separar_rechazos(df, ~q.validar_ruts(df["rut"]), "rut_invalido"); rechazos.append(r)

    df["codigo_campana"] = q.normalizar_texto(df["codigo_campana"], mayusculas=True)
    df["sku"] = q.normalizar_sku(df["sku_promocionado"])
    df["canal"] = q.canal_canonico(df["canal"])
    df["respondio"] = (q.normalizar_texto(df["respondio"], mayusculas=True) == "SI").astype(int)
    df["rut_anonimizado"] = q.anonimizar_serie(df["rut"], exigir_rut_salt())      # el RUT no sigue más allá de este punto
    df = df.drop(columns=["rut", "sku_promocionado"])

    df, r = q.separar_rechazos(df, df.duplicated(["codigo_campana", "rut_anonimizado"], keep="first"), "contacto_duplicado"); rechazos.append(r)

    dim_campana = db.leer_tabla(engine, "dim_campana")
    dim_campana = dim_campana[dim_campana["campana_key"] != -1].set_index("codigo_campana")
    df, r = q.separar_rechazos(df, ~df["codigo_campana"].isin(dim_campana.index), "campana_desconocida"); rechazos.append(r)
    inicio = pd.to_datetime(df["codigo_campana"].map(dim_campana["fecha_inicio"]))
    fin = pd.to_datetime(df["codigo_campana"].map(dim_campana["fecha_fin"]))
    df, r = q.separar_rechazos(df, (df["fecha_envio"] < inicio) | (df["fecha_envio"] > fin), "fuera_de_campana"); rechazos.append(r)

    skus = set(db.leer_tabla(engine, "dim_producto")["sku"])
    df, r = q.separar_rechazos(df, ~df["sku"].isin(skus), "sku_desconocido"); rechazos.append(r)
    df, r = q.separar_rechazos(df, df["canal"].isna(), "canal_desconocido"); rechazos.append(r)
    return df, q.consolidar_rechazos(*rechazos)

### B.3 Claves sustitutas
Cliente y producto toman la **versión vigente en la fecha de envío** (SCD2). Un cliente sin ficha no puede ser contactado en el modelo.

In [ ]:
CLAVES_HECHO = ["campana_key", "cliente_key", "producto_key", "canal_key", "fecha_key"]

def asignar_claves(c: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    v = c.copy()
    v["fecha_key"] = v["fecha_envio"].dt.strftime("%Y%m%d").astype(int)
    v["campana_key"] = v["codigo_campana"].map(db.mapa_claves(engine, "dim_campana", "codigo_campana", "campana_key")).astype(int)
    v["canal_key"] = v["canal"].map(db.mapa_claves(engine, "dim_canal", "canal", "canal_key")).fillna(-1).astype(int)
    v["producto_key"] = db.asignar_clave_scd2(v, db.leer_tabla(engine, "dim_producto"), "sku", "fecha_envio", "producto_key")
    v["cliente_key"] = db.asignar_clave_scd2(v, db.leer_tabla(engine, "dim_cliente"), "rut_anonimizado", "fecha_envio", "cliente_key")
    v, r1 = q.separar_rechazos(v, v["cliente_key"] == -1, "cliente_no_registrado")
    v, r2 = q.separar_rechazos(v, v["producto_key"] == -1, "producto_no_vigente")
    return v, q.consolidar_rechazos(r1, r2)

### B.4 Atribución de ventas y costo asignado
- **Venta atribuida:** ventas netas (sin devoluciones) del mismo cliente y mismo producto dentro de la ventana; cada línea de venta se atribuye **a lo más a un contacto** (el más reciente), así K5 no cuenta dos veces.
- **Costo asignado:** presupuesto de la campaña ÷ clientes contactados (partes iguales; la suma por campaña cierra exacto con el presupuesto).

In [ ]:
def atribuir_ventas(contactos: pd.DataFrame) -> pd.DataFrame:
    consulta = """
    SELECT f.id_transaccion, f.linea, d.fecha, p.sku, c.rut_anonimizado, f.monto_neto
    FROM fact_ventas f
    JOIN dim_fecha d    ON d.fecha_key = f.fecha_key
    JOIN dim_producto p ON p.producto_key = f.producto_key
    JOIN dim_cliente c  ON c.cliente_key = f.cliente_key
    WHERE f.devolucion = 0 AND f.producto_key <> -1 AND f.cliente_key <> -1
    """
    with engine.connect() as conn:
        ventas = pd.read_sql(consulta, conn)
    ventas["fecha"] = pd.to_datetime(ventas["fecha"])

    cruce = contactos[CLAVES_HECHO + ["rut_anonimizado", "sku", "fecha_envio"]].merge(ventas, on=["rut_anonimizado", "sku"])
    en_ventana = (cruce["fecha"] >= cruce["fecha_envio"]) & (cruce["fecha"] <= cruce["fecha_envio"] + pd.Timedelta(days=VENTANA_ATRIBUCION_DIAS))
    cruce = cruce[en_ventana]
    # Último contacto: si la venta cae en la ventana de varias campañas, se la lleva la de envío más reciente
    cruce = (cruce.sort_values(["fecha_envio", "campana_key"], ascending=[False, False])
                  .drop_duplicates(["id_transaccion", "linea"], keep="first"))
    assert not cruce.duplicated(["id_transaccion", "linea"]).any(), "una venta quedó atribuida a más de un contacto"
    return (cruce.groupby(CLAVES_HECHO, as_index=False)["monto_neto"].sum()
                 .rename(columns={"monto_neto": "monto_venta_atribuida"}))


def construir_hechos(contactos: pd.DataFrame) -> pd.DataFrame:
    h = contactos.merge(atribuir_ventas(contactos), on=CLAVES_HECHO, how="left")
    h["monto_venta_atribuida"] = h["monto_venta_atribuida"].fillna(0.0)
    h["compro"] = (h["monto_venta_atribuida"] > 0).astype(int)
    h["contactado"] = 1
    dim_campana = db.leer_tabla(engine, "dim_campana").set_index("campana_key")
    presupuesto = h["campana_key"].map(dim_campana["presupuesto"])
    h["costo_asignado"] = (presupuesto / h.groupby("campana_key")["campana_key"].transform("size")).round(2)
    # Ajuste de centavos: la suma por campaña debe cerrar exacto con el presupuesto
    residuo = dim_campana["presupuesto"] - h.groupby("campana_key")["costo_asignado"].sum()
    primera = h.groupby("campana_key").head(1).index
    h.loc[primera, "costo_asignado"] = (h.loc[primera, "costo_asignado"] + h.loc[primera, "campana_key"].map(residuo)).round(2)
    return h

### B.5 Load y ejecución
Carga idempotente por la clave compuesta del grano.

In [ ]:
COLUMNAS_HECHOS = [*CLAVES_HECHO, "contactado", "respondio", "compro", "monto_venta_atribuida", "costo_asignado"]

def cargar_hechos(hechos: pd.DataFrame, rechazos: pd.DataFrame) -> dict:
    assert set(COLUMNAS_HECHOS) <= set(hechos.columns), "construir_hechos() no devolvió las columnas esperadas"
    db.registrar_rechazos(engine, PROCESO, rechazos)
    return db.insertar_si_no_existe(engine, schema.fact_campanas, hechos[COLUMNAS_HECHOS], CLAVES_HECHO)

In [ ]:
with db.Ejecucion(engine, PROCESO) as run:
    contactos_crudo = extraer_contactos()
    run.leidas = len(contactos_crudo)
    contactos_validos, rech_calidad = transformar_contactos(contactos_crudo)
    medallion.guardar_capa(contactos_validos, "silver", "campanas_contactos")
    contactos_con_claves, rech_claves = asignar_claves(contactos_validos)
    rechazos = q.consolidar_rechazos(rech_calidad, rech_claves)
    hechos = construir_hechos(contactos_con_claves)
    resumen = cargar_hechos(hechos, rechazos)
    run.cargadas = resumen["insertadas"]
    run.rechazadas = len(rechazos)
resumen

## 5. Validación

In [ ]:
def validar_hechos():
    n_leidas, n_cargables, n_rech = len(contactos_crudo), len(hechos), len(rechazos)
    assert n_leidas == n_cargables + n_rech, f"No concilia: {n_leidas} leídas ≠ {n_cargables} cargables + {n_rech} rechazadas"
    cargados = db.leer_tabla(engine, "fact_campanas")
    assert len(cargados) == n_cargables, "lo cargado no coincide con lo esperado"
    assert ((cargados["compro"] == 1) == (cargados["monto_venta_atribuida"] > 0)).all(), "compro inconsistente con la venta atribuida"
    assert (cargados["respondio"].isin([0, 1])).all() and (cargados["contactado"] == 1).all()
    # El costo asignado de cada campaña debe sumar exactamente su presupuesto
    dim = db.leer_tabla(engine, "dim_campana").set_index("campana_key")
    costo = cargados.groupby("campana_key")["costo_asignado"].sum()
    assert ((costo - dim.loc[costo.index, "presupuesto"]).abs() < 0.01).all(), "el costo asignado no suma el presupuesto"
    print(f"Conciliación OK · {n_leidas} = {n_cargables} + {n_rech} · el costo asignado cierra con el presupuesto en {len(costo)} campañas")

validar_hechos()

In [ ]:
# Idempotencia: volver a cargar los mismos hechos no inserta nada
filas_antes = len(db.leer_tabla(engine, "fact_campanas"))
r2 = cargar_hechos(hechos, pd.DataFrame(columns=q.COLUMNAS_RECHAZO))
assert len(db.leer_tabla(engine, "fact_campanas")) == filas_antes and r2["insertadas"] == 0
print("Idempotencia OK:", r2)

## 6. Evidencia
K5 por campaña: respuesta, conversión, ingreso atribuido, inversión y ROAS.

In [ ]:
consulta_k5 = """
SELECT c.codigo_campana, c.nombre_campana,
       SUM(f.contactado) AS contactados, SUM(f.respondio) AS respondieron, SUM(f.compro) AS compraron,
       ROUND(100.0 * SUM(f.respondio) / SUM(f.contactado), 1) AS respuesta_pct,
       ROUND(100.0 * SUM(f.compro) / SUM(f.contactado), 1) AS conversion_pct,
       SUM(f.monto_venta_atribuida) AS ingreso_atribuido, SUM(f.costo_asignado) AS inversion,
       ROUND(1.0 * SUM(f.monto_venta_atribuida) / NULLIF(SUM(f.costo_asignado), 0), 2) AS roas
FROM fact_campanas f
JOIN dim_campana c ON c.campana_key = f.campana_key
GROUP BY c.codigo_campana, c.nombre_campana
ORDER BY c.codigo_campana
"""
with engine.connect() as conn:
    display(pd.read_sql(consulta_k5, conn))
rech = db.leer_tabla(engine, "auditoria_rechazos")
rech[rech["proceso"].isin(["fact_campanas", "dim_campana"])].groupby(["proceso", "motivo_rechazo"]).size().rename("filas").to_frame()

## 7. Decisiones de diseño (completar)
- **Atribución por último contacto dentro de 14 días:** evita contar una venta en dos campañas. Otras reglas (primer contacto, reparto proporcional) cambian el ROAS de cada campaña; la ventana es el parámetro `VENTANA_ATRIBUCION_DIAS`.
- **`costo_asignado` en partes iguales:** hace que la suma por campaña cierre con el presupuesto; no distingue contactos más caros que otros.
- **Carga idempotente, no incremental:** si `fact_ventas` cambia, hay que borrar y recargar `fact_campanas`, porque la venta atribuida depende de las ventas ya cargadas.
- **`respondio` lo informa Marketing:** es posible comprar sin haber respondido; por eso K5 usa `compro`, que sale de las ventas.
- **ROAS ≠ ROI:** ROAS = ingreso atribuido ÷ inversión (no descuenta el costo de los productos vendidos).